# Step 0~2 전처리 (Colab · GPU 불필요)

원본 STT txt → `raw.jsonl` → `merged.jsonl` (규칙 기반). 토큰 없이 **src 폴더 업로드** 방식.

### 사전 준비 (Drive에 2개 업로드)
`MyDrive/lecture-analyzer/` 폴더를 만들고 그 안에 넣기:
1. 로컬 repo의 **`src/` 폴더** 통째로 → `MyDrive/lecture-analyzer/src/`
2. 원본 데이터 **`강의 스크립트/` 폴더**(txt 15개) → `MyDrive/lecture-analyzer/data_scripts/`

> ⚠️ 원본 데이터는 **개인 Drive(비공개)** 까지만. git/공개 업로드 금지.
> 이 노트북은 GPU 불필요 → CPU 런타임으로 돌려도 됨(정제 단계만 A100 필요).

## 1. Drive 마운트 + 코드 경로 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
DRIVE = '/content/drive/MyDrive/lecture-analyzer'
sys.path.insert(0, DRIVE)   # 업로드한 src/ 를 import 경로에 추가
import src.config  # 임포트 되면 코드 연결 성공
print('코드 연결 OK')

## 2. 입력 경로 설정 + 확인
업로드한 txt 폴더 경로를 맞춘다(아래 SCRIPT_DIR).

In [ ]:
from pathlib import Path
SCRIPT_DIR = Path(DRIVE) / 'data_scripts'      # ← 원본 txt 15개 올린 폴더
OUT = Path(DRIVE)                              # 산출물 저장 위치(Drive)
txts = sorted(SCRIPT_DIR.glob('*.txt'))
assert txts, f'txt를 찾을 수 없음: {SCRIPT_DIR} (업로드 경로 확인)'
print(f'입력 txt {len(txts)}개 확인:', txts[0].name, '...')

## 3. Step 1 — 파싱 → raw.jsonl

In [ ]:
from src.preprocess.parse import write_raw_jsonl
s1 = write_raw_jsonl(OUT / 'raw.jsonl', script_dir=SCRIPT_DIR)
print(s1)

## 4. Step 2 — 화자 매핑 + 발화 병합 → merged.jsonl + speaker_map.json

In [ ]:
from src.preprocess.merge import run_step2
s2 = run_step2(OUT / 'raw.jsonl', OUT)
print(s2)

## 5. manifest 기록 (재현성) + 결과 미리보기

In [ ]:
from src.manifest import write_manifest
from src import config
write_manifest(OUT / 'manifest_preprocess.json', step='preprocess(step0-2)',
    params={'merge_gap_sec': config.MERGE_GAP_SEC,
            'merge_max_block_sec': config.MERGE_MAX_BLOCK_SEC,
            'merge_max_block_chars': config.MERGE_MAX_BLOCK_CHARS},
    stats={'step1': s1, 'step2': s2}, inputs=txts)

import json
with (OUT / 'merged.jsonl').open(encoding='utf-8') as f:
    first = json.loads(f.readline())
print('merged 첫 블록:', first['speaker_role'], first['start_time'], '~', first['end_time'],
      '| 글자', len(first['text']))
print('\n✅ 전처리 완료. 다음: 02_refine_colab.ipynb 에서 merged.jsonl 로 정제(Solar) 실행')